# 08 · Reserved-slot hybrid: protecting ALS while recovering rare movies

## Project context

This notebook follows the experiments in `07_hybrid_recommender.ipynb`. The current strongest reference is **implicit ALS alone**. The frozen **content-v1** representation consists of MovieLens genres + release decade. **TMDB content-v2** extends that representation with TMDB genres, keywords, overview, director, and cast.

Previous results found that ordinary rank-percentile blending (`90% ALS + 10% content`) underperformed ALS on full-catalogue validation. In the temporally aligned validation window, ALS had overall View A NDCG@10 around **0.081462**, while the tested blends were much lower. On the tail, ALS could rank relevant rare movies when only tail items competed (View B), but almost none reached the top 10 when all catalogue movies competed (View A).

The next hypothesis is architectural, not another score-weight sweep:

> Keep the first `10-r` recommendations strictly ranked by ALS, then reserve up to `r` final positions for unseen, tail-catalogue items selected by a content profile. If content cannot find enough positive-similarity candidates, fill the remaining positions with the next ALS-ranked items.

## What this experiment compares

- `ALS_only`: unchanged 10-item ALS reference (popularity fallback for users with no positive training history).
- `ALS_tail_r1`: 9 ALS picks + 1 tail item chosen by ALS among items with learned factors. This is a control for reserving exposure without switching to content ranking.
- `content_v1_r1/r2/r3`: content-v1 picks fill 1, 2, or 3 reserved tail slots.
- `tmdb_v2_r1/r2/r3`: the same reserved-slot design using TMDB content-v2.

All final lists have length `K=10`; recommendations are deduplicated, exclude training-seen movies, and preserve ALS order in the primary slots. This is a validation experiment only. **The test split is not used to train, tune, select, or report results in this notebook.**

## 1 · Predefined protocol and decision rule

### Candidate and evaluation definitions

- Positive interaction / relevance: rating `>= 4.0`.
- Tail pool: movies with **fewer than 20 positive training interactions**, computed from `train` only and fixed for every model.
- Strict-cold pool: movies with **zero positive training interactions**.
- Warm-tail pool: movies with 1–19 positive training interactions.
- Validation target window: positive validation interactions at or after `2016-01-01 00:00:00 UTC`, matching notebook 07's full-catalogue comparison. Overall, tail, strict-cold, and warm-tail metrics are computed from that same window and from the same recommendations per user and seed.
- View A: every movie in the 17,241-item catalogue competes for top 10. For tail/strict-cold/warm-tail metrics, only targets in that slice count as hits; the candidate pool remains the full catalogue.

### Grid (fixed before running)

| Candidate | Reserved slots | Reserved-item source |
|---|---:|---|
| ALS-only | 0 | ALS / existing popularity fallback |
| ALS-tail control | 1 | Highest-ranked eligible ALS tail item, outside the first 9 |
| content-v1 | 1, 2, 3 | Highest positive profile-similarity among unseen tail candidates |
| TMDB-v2 | 1, 2, 3 | Same content-based candidate rule using TMDB-v2 features |

### Guardrail and selection

- Overall NDCG@10 must be at least **98% of the ALS-only overall View A NDCG@10**. The threshold is calculated from the ALS baseline in this exact run (the previous reference was approximately `0.079833`).
- Among reserved-slot candidates meeting that point-estimate guardrail, select the one with the highest **tail View A NDCG@10**.
- Report a paired user-level bootstrap interval for the overall guardrail margin `candidate NDCG@10 - 0.98 × ALS NDCG@10`. A passing point estimate with an interval crossing zero means preservation is uncertain; it is not formal proof of non-inferiority.
- Also report paired intervals for overall, tail, strict-cold, and warm-tail NDCG@10 and Recall@10, plus operational diagnostics (injected hits, displaced ALS hits, users gaining/losing hits, tail share, coverage, and fallback rate).

### Leakage controls

Content profiles use **training interactions only**, and only ratings `>= 4.0`. Candidate pools and popularity fallback are built from training data. Validation selects among the predefined candidates; no test result is consulted. The cutoff aligns the evaluated validation targets across slices; it is not a claim that the entire training split is globally earlier than 2016.

In [1]:
# 2 · Load the existing split and saved feature matrices (no API calls)

import os
import sys
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

# Resolve project root, then run from notebooks/ because recutils.load_split uses
# the same relative-path convention as the earlier project notebooks.
_START = Path.cwd().resolve()
_PROJECT_ROOT = None
for _p in [_START, *_START.parents]:
    if (_p / "scripts" / "recutils.py").exists() and (_p / "data").exists():
        _PROJECT_ROOT = _p
        break
if _PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing scripts/recutils.py and data/. "
        "Open this notebook from your movie-recommender project."
    )
_NOTEBOOK_DIR = _PROJECT_ROOT / "notebooks"
if _NOTEBOOK_DIR.exists() and Path.cwd().resolve() != _NOTEBOOK_DIR.resolve():
    os.chdir(_NOTEBOOK_DIR)
sys.path.insert(0, str(_PROJECT_ROOT / "scripts"))

from recutils import load_split

NAME, K, REL, SEED = "sample_10pct", 10, 4.0, 42
N_TIE_SEEDS = 10
N_BOOT = 1000
CUTOFF_TS = int(pd.Timestamp("2016-01-01 00:00:00", tz="UTC").timestamp())
RETENTION_FRACTION = 0.98
BATCH_SIZE = 96

S = load_split(NAME)
train = S["train"].copy()
val = S["val"].copy()
# Deliberately do not assign or inspect S["test"].
item_map = S["item_map"].sort_values("i").reset_index(drop=True)
user_map = S["user_map"].sort_values("u").reset_index(drop=True)
n_users = int(user_map["u"].max()) + 1
n_items = int(item_map["i"].max()) + 1

feature_dir = _PROJECT_ROOT / "data" / "interim" / "hybrid_features"
tmdb_dir = _PROJECT_ROOT / "data" / "processed" / NAME

v1_path = feature_dir / "content_v1.npz"
tmdb_path = tmdb_dir / "tmdb_content_v2.npz"
tmdb_index_path = tmdb_dir / "tmdb_content_v2_index.parquet"

for _path in [v1_path, tmdb_path, tmdb_index_path]:
    if not _path.exists():
        raise FileNotFoundError(
            f"Required saved artifact not found: {_path}\n"
            "Generate the content-v1 and TMDB-v2 artifacts in the earlier notebooks first."
        )

X_v1 = sp.load_npz(v1_path).tocsr().astype(np.float32)
X_tmdb_v2 = sp.load_npz(tmdb_path).tocsr().astype(np.float32)
tmdb_item_index = pd.read_parquet(tmdb_index_path).sort_values("i").reset_index(drop=True)

assert X_v1.shape == (n_items, 31), f"Unexpected content-v1 shape: {X_v1.shape}"
assert X_tmdb_v2.shape[0] == n_items, f"Unexpected TMDB shape: {X_tmdb_v2.shape}"
assert np.array_equal(item_map["i"].to_numpy(), np.arange(n_items))
assert np.array_equal(tmdb_item_index["i"].to_numpy(), item_map["i"].to_numpy())
assert np.array_equal(tmdb_item_index["movieId"].to_numpy(), item_map["movieId"].to_numpy())
assert np.isfinite(X_v1.data).all() and np.isfinite(X_tmdb_v2.data).all()

# Build the user-item seen matrix from ALL training ratings, not just positive ratings.
seen_train = sp.csr_matrix(
    (
        np.ones(len(train), dtype=np.float32),
        (train["u"].to_numpy(dtype=np.int64), train["i"].to_numpy(dtype=np.int64)),
    ),
    shape=(n_users, n_items),
)
seen_train.sum_duplicates()
seen_train.data[:] = 1.0

# Positive training counts define the fixed tail/strict-cold/warm-tail pools.
train_positive = train.loc[train["rating"] >= REL]
n_pos = np.bincount(train_positive["i"].to_numpy(dtype=np.int64), minlength=n_items)
tail_mask = n_pos < 20
strict_cold_mask = n_pos == 0
warm_tail_mask = (n_pos >= 1) & (n_pos < 20)
tail_item_idx = np.flatnonzero(tail_mask)

# Existing popularity fallback: count of all train interactions per item.
popularity_scores = np.bincount(
    train["i"].to_numpy(dtype=np.int64), minlength=n_items
).astype(np.float64)
u_has_pos = np.bincount(
    train_positive["u"].to_numpy(dtype=np.int64), minlength=n_users
) > 0

# Same post-cutoff positive validation window for all scopes.
assert "timestamp" in val.columns, "Validation split has no timestamp column."
val_ts = pd.to_numeric(val["timestamp"], errors="raise")
val_window = val.loc[
    (val["rating"] >= REL) & (val_ts >= CUTOFF_TS),
    ["u", "i", "timestamp"],
].drop_duplicates(["u", "i"]).copy()

def relevant_dict(frame):
    return {
        int(user): set(group["i"].astype(int).tolist())
        for user, group in frame.groupby("u", sort=False)
    }

relevant_by_scope = {
    "overall": relevant_dict(val_window),
    "tail": relevant_dict(val_window.loc[tail_mask[val_window["i"].to_numpy(dtype=int)]]),
    "strict_cold": relevant_dict(val_window.loc[strict_cold_mask[val_window["i"].to_numpy(dtype=int)]]),
    "warm_tail": relevant_dict(val_window.loc[warm_tail_mask[val_window["i"].to_numpy(dtype=int)]]),
}

eval_users = np.asarray(sorted(relevant_by_scope["overall"]), dtype=np.int64)
scope_users = {
    scope: np.asarray(sorted(rel.keys()), dtype=np.int64)
    for scope, rel in relevant_by_scope.items()
}
assert len(eval_users) > 0 and len(scope_users["tail"]) > 0

# Sanity check that validation target pairs are not seen in train.
_target_u = val_window["u"].to_numpy(dtype=np.int64)
_target_i = val_window["i"].to_numpy(dtype=np.int64)
_seen_target_values = np.asarray(seen_train[_target_u, _target_i]).ravel()
assert not _seen_target_values.any(), "Some validation targets are also in training history."

print(f"Project root: {_PROJECT_ROOT}")
print(f"Users: {n_users:,} | Catalogue movies: {n_items:,}")
print(f"Train rows: {len(train):,} | Validation rows: {len(val):,}")
print(f"content-v1 matrix: {X_v1.shape}, nnz={X_v1.nnz:,}")
print(f"TMDB-v2 matrix: {X_tmdb_v2.shape}, nnz={X_tmdb_v2.nnz:,}")
print(f"Tail pool (<20 positives): {int(tail_mask.sum()):,} movies")
print(f"Strict-cold pool (0 positives): {int(strict_cold_mask.sum()):,} movies")
print(f"Warm-tail pool (1–19 positives): {int(warm_tail_mask.sum()):,} movies")
print(f"Validation window: {pd.Timestamp(CUTOFF_TS, unit='s', tz='UTC')} onward")
for _scope, _rel in relevant_by_scope.items():
    print(f"{_scope:12s}: {sum(map(len, _rel.values())):,} relevant pairs, {len(_rel):,} users")
print("Setup/alignment checks passed. Test outcomes have not been accessed.")

Project root: C:\Users\Lenovo\Documents\movie-recommender
Users: 20,093 | Catalogue movies: 17,241
Train rows: 2,919,680 | Validation rows: 137,197
content-v1 matrix: (17241, 31), nnz=53,959
TMDB-v2 matrix: (17241, 91328), nnz=873,444
Tail pool (<20 positives): 11,597 movies
Strict-cold pool (0 positives): 1,095 movies
Warm-tail pool (1–19 positives): 10,502 movies
Validation window: 2016-01-01 00:00:00+00:00 onward
overall     : 25,406 relevant pairs, 5,661 users
tail        : 1,585 relevant pairs, 948 users
strict_cold : 52 relevant pairs, 50 users
warm_tail   : 1,533 relevant pairs, 931 users
Setup/alignment checks passed. Test outcomes have not been accessed.


## 3 · Fit the selected implicit ALS model on training data only

Use the same ALS configuration as notebook 07: positive training interactions (`rating >= 4.0`), 64 factors, `alpha=5`, regularization `lambda=80`, 10 alternating iterations, and fixed random seed. The model is refit here so this notebook can run independently. No validation interactions are used in fitting.

In [2]:
def _implicit_half_step(M, Y, lam):
    """One ALS half-step for implicit feedback; M stores confidence minus one."""
    n_rows = M.shape[0]
    k = Y.shape[1]
    YtY = Y.T @ Y
    reg = lam * np.eye(k)
    out = np.zeros((n_rows, k), dtype=np.float64)
    indptr, indices, data = M.indptr, M.indices, M.data

    for row in range(n_rows):
        start, end = indptr[row], indptr[row + 1]
        if start == end:
            continue
        item_factors = Y[indices[start:end]]
        confidence_minus_one = data[start:end]
        A = (
            YtY
            + (item_factors.T * confidence_minus_one) @ item_factors
            + reg
        )
        b = item_factors.T @ (confidence_minus_one + 1.0)
        out[row] = np.linalg.solve(A, b)
    return out


def fit_implicit_als_train_only(df, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED):
    """Fit implicit ALS to the supplied train split only."""
    d = df.loc[df["rating"] >= REL, ["u", "i"]].drop_duplicates(["u", "i"])
    C = sp.csr_matrix(
        (
            np.full(len(d), float(alpha), dtype=np.float64),
            (d["u"].to_numpy(dtype=np.int64), d["i"].to_numpy(dtype=np.int64)),
        ),
        shape=(n_users, n_items),
    )
    C.sum_duplicates()
    C.eliminate_zeros()
    Ct = C.T.tocsr()
    rng = np.random.default_rng(seed)
    Q = rng.normal(0, 0.1, size=(n_items, k))
    P = np.zeros((n_users, k), dtype=np.float64)
    start_time = time.time()

    for iteration in range(1, n_iter + 1):
        P = _implicit_half_step(C, Q, lam)
        Q = _implicit_half_step(Ct, P, lam)
        if iteration == 1 or iteration % 2 == 0 or iteration == n_iter:
            print(f"Iteration {iteration:02d}/{n_iter} | elapsed {time.time() - start_time:.1f}s")

    return {"P": P, "Q": Q}

als_model = fit_implicit_als_train_only(
    train, k=64, alpha=5.0, lam=80.0, n_iter=10, seed=SEED
)
P_als = als_model["P"].astype(np.float32)
Q_als = als_model["Q"].astype(np.float32)
item_has_factors = np.abs(Q_als).sum(axis=1) > 0

assert P_als.shape == (n_users, 64)
assert Q_als.shape == (n_items, 64)
assert np.isfinite(P_als).all() and np.isfinite(Q_als).all()
assert int((~item_has_factors).sum()) == int(strict_cold_mask.sum()), (
    "Zero-factor items differ from items with zero positive training interactions; inspect ALS implementation."
)

print("\nALS train-only fit complete.")
print("Users without positive history:", int((~u_has_pos).sum()))
print("Movies without ALS factors:", int((~item_has_factors).sum()))

Iteration 01/10 | elapsed 3.3s
Iteration 02/10 | elapsed 6.4s
Iteration 04/10 | elapsed 12.8s
Iteration 06/10 | elapsed 19.1s
Iteration 08/10 | elapsed 25.3s
Iteration 10/10 | elapsed 31.4s

ALS train-only fit complete.
Users without positive history: 34
Movies without ALS factors: 1095


## 4 · Build train-only user content profiles

For each user, average the item vectors of movies rated at least 4.0 in the training split. The profile is then scored against tail-item vectors using a dot product. This is a content similarity score, not a predicted rating or probability.

We keep only positive-similarity tail candidates for injection. If a profile has no signal or cannot retrieve enough unseen tail candidates after deduplication, the missing reserved slots are filled by the next eligible ALS-ranked items. Zero-positive-history users retain the existing popularity fallback and are not injected with content recommendations.

In [3]:
# Build the train-positive user-item matrix once; reused for both content representations.
liked_train = train.loc[train["rating"] >= REL, ["u", "i"]].drop_duplicates(["u", "i"])
L_profile = sp.csr_matrix(
    (
        np.ones(len(liked_train), dtype=np.float32),
        (liked_train["u"].to_numpy(dtype=np.int64), liked_train["i"].to_numpy(dtype=np.int64)),
    ),
    shape=(n_users, n_items),
)
L_profile.sum_duplicates()
L_profile.data[:] = 1.0
n_profile_items = np.asarray(L_profile.sum(axis=1)).ravel().astype(np.float32)
profile_inverse = np.divide(
    1.0,
    n_profile_items,
    out=np.zeros_like(n_profile_items, dtype=np.float32),
    where=n_profile_items > 0,
)
profile_scale = sp.diags(profile_inverse, format="csr", dtype=np.float32)

content_items = {
    "content_v1": X_v1,
    "tmdb_v2": X_tmdb_v2,
}
profiles = {}
tail_feature_matrices = {}
profile_has_signal = {}
tail_has_features = {}

for name, X in content_items.items():
    print(f"Building train-only profiles for {name} ...")
    profiles[name] = (profile_scale @ (L_profile @ X)).tocsr().astype(np.float32)
    profile_has_signal[name] = np.asarray(profiles[name].getnnz(axis=1) > 0).ravel()
    tail_feature_matrices[name] = X[tail_item_idx].tocsr()
    tail_has_features[name] = np.asarray(
        tail_feature_matrices[name].getnnz(axis=1) > 0
    ).ravel()
    print(
        f"{name}: profiles={profiles[name].shape}, profile nnz={profiles[name].nnz:,}, "
        f"tail candidates with any features={int(tail_has_features[name].sum()):,}/{len(tail_item_idx):,}"
    )

print("Profiles use only train rows with rating >=", REL)
print("Users with positive history:", int(u_has_pos.sum()))

Building train-only profiles for content_v1 ...
content_v1: profiles=(20093, 31), profile nnz=387,044, tail candidates with any features=11,587/11,597
Building train-only profiles for tmdb_v2 ...
tmdb_v2: profiles=(20093, 91328), profile nnz=56,949,491, tail candidates with any features=11,592/11,597
Profiles use only train rows with rating >= 4.0
Users with positive history: 20059


## 5 · Recommendation construction: reserve slots, don't blend scores

The algorithm builds the ALS ranking once per user/seed. For each candidate:

1. Preserve the top `K-r` eligible ALS items in their original order.
2. Identify unseen tail items (`n_pos < 20`) with positive content-profile similarity, excluding the preserved ALS items.
3. Select up to `r` distinct tail items with the highest profile similarity and append them to the reserved section.
4. If fewer than `r` valid content picks exist, fill the remaining positions with the next best eligible ALS recommendations. Never duplicate an item or recommend a training-seen item.

The ALS-tail control follows the same primary-slot rule but chooses the reserved item by ALS score among unseen tail movies with learned factors. For users with no positive training history, every method keeps the popularity fallback unchanged.

In [4]:
# Candidate definitions, in a fixed grid.
R_VALUES = (1, 2, 3)
CANDIDATES = ["ALS_only", "ALS_tail_r1"]
for _name in ("content_v1", "tmdb_v2"):
    CANDIDATES.extend([f"{_name}_r{r}" for r in R_VALUES])


def _seen_items_for_user(user):
    start, end = seen_train.indptr[int(user)], seen_train.indptr[int(user) + 1]
    return seen_train.indices[start:end]


def _fill_from_als(primary_items, injected_items, als_order, k=K):
    """Keep primary items and injections; use next ALS items to reach length k."""
    result = []
    selected = set()
    for item in list(primary_items) + list(injected_items):
        item = int(item)
        if item not in selected:
            selected.add(item)
            result.append(item)
    for item in als_order:
        item = int(item)
        if item not in selected:
            selected.add(item)
            result.append(item)
            if len(result) >= k:
                break
    return result[:k]


def _top_r_content_candidates(candidate_ids, candidate_scores, excluded_ids, r):
    """Choose up to r highest positive-similarity candidate IDs after exclusions."""
    if len(candidate_ids) == 0:
        return []
    keep = ~np.isin(candidate_ids, excluded_ids)
    ids = candidate_ids[keep]
    scores = candidate_scores[keep]
    if len(ids) == 0:
        return []
    take = min(int(r), len(ids))
    if len(ids) > take:
        positions = np.argpartition(scores, len(scores) - take)[-take:]
    else:
        positions = np.arange(len(ids))
    positions = positions[np.argsort(scores[positions], kind="stable")[::-1]]
    return ids[positions].astype(int).tolist()


def _rank_als_batch(batch_users, rng_als):
    """Return complete eligible ALS rankings for this batch, excluding seen movies."""
    scores = (P_als[batch_users] @ Q_als.T).astype(np.float64)
    for row_idx, user in enumerate(batch_users):
        if u_has_pos[int(user)]:
            scores[row_idx, ~item_has_factors] = -np.inf
        else:
            # Existing no-positive-history policy: popularity fallback.
            scores[row_idx, :] = popularity_scores
        seen = _seen_items_for_user(int(user))
        scores[row_idx, seen] = -np.inf

    # Convert exact ties into reproducible, negligible order differences.
    scores += rng_als.random(scores.shape) * 1e-10
    order_matrix = np.argsort(-scores, axis=1, kind="stable")
    orders = []
    for row_idx in range(len(batch_users)):
        order = order_matrix[row_idx]
        valid = np.isfinite(scores[row_idx, order])
        orders.append(order[valid].astype(np.int32))
    return orders


def _metrics_for_recommendation(recommendation, relevant):
    """Per-user NDCG, Recall, hit rate, and hit count. Empty target sets are NaN."""
    if not relevant:
        return {"ndcg": np.nan, "recall": np.nan, "hit_rate": np.nan, "hit_count": np.nan}
    discounts = 1.0 / np.log2(np.arange(2, K + 2))
    hits = np.fromiter((int(item in relevant) for item in recommendation), dtype=np.float64)
    idcg = discounts[:min(len(relevant), K)].sum()
    dcg = (hits[:len(discounts)] * discounts[:len(hits)]).sum()
    return {
        "ndcg": float(dcg / idcg) if idcg else 0.0,
        "recall": float(hits.sum() / len(relevant)),
        "hit_rate": float(hits.sum() > 0),
        "hit_count": float(hits.sum()),
    }


def _generate_batch_slates(batch_users, seed, rng_als, rng_content):
    """Return each candidate's slate and per-user diagnostics for one batch."""
    als_orders = _rank_als_batch(batch_users, rng_als)

    # Compute user-profile similarity to only the tail candidate pool.
    tail_similarities = {}
    valid_content_ids = {}
    valid_content_scores = {}
    for name in content_items:
        raw = (profiles[name][batch_users] @ tail_feature_matrices[name].T).toarray().astype(np.float64)
        tail_similarities[name] = raw

    batch_slates = {name: [] for name in CANDIDATES}
    batch_injected = {name: [] for name in CANDIDATES}
    batch_diagnostics = {name: [] for name in CANDIDATES}

    for row_idx, user_value in enumerate(batch_users):
        user = int(user_value)
        als_order = als_orders[row_idx]
        als_top10 = als_order[:K].astype(int).tolist()
        if len(als_top10) < K:
            raise RuntimeError(f"User {user} has fewer than {K} eligible unseen ALS/fallback candidates.")
        seen = _seen_items_for_user(user)
        seen_set = set(map(int, seen))
        overall_rel = relevant_by_scope["overall"].get(user, set())
        tail_rel = relevant_by_scope["tail"].get(user, set())
        strict_cold_rel = relevant_by_scope["strict_cold"].get(user, set())
        warm_tail_rel = relevant_by_scope["warm_tail"].get(user, set())

        # Generate one noisy candidate ranking per content representation and user.
        # Reuse the same tie noise across r=1/2/3 for fair nested comparisons.
        user_content_rankings = {}
        for name in content_items:
            raw_scores = tail_similarities[name][row_idx]
            candidate_pos = np.flatnonzero(raw_scores > 0.0)
            ids = tail_item_idx[candidate_pos].astype(np.int32)
            scores = raw_scores[candidate_pos].copy()
            if len(scores):
                scores += rng_content[name].random(len(scores)) * 1e-10
            user_content_rankings[name] = (ids, scores)

        # Strategy 1: unchanged ALS-only ranking.
        batch_slates["ALS_only"].append(als_top10)
        batch_injected["ALS_only"].append([])

        # Strategy 2: 9 ALS picks + 1 ALS-ranked eligible tail pick.
        control_injected = []
        if u_has_pos[user]:
            control_base = als_order[:K - 1].astype(int).tolist()
            control_base_set = set(control_base)
            for item in als_order:
                item = int(item)
                if tail_mask[item] and item_has_factors[item] and item not in control_base_set:
                    control_injected = [item]
                    break
            control_slate = _fill_from_als(control_base, control_injected, als_order, K)
        else:
            # Do not override the established popularity fallback for no-history users.
            control_slate = als_top10
        batch_slates["ALS_tail_r1"].append(control_slate)
        batch_injected["ALS_tail_r1"].append(control_injected)

        # Strategies 3–8: reserve r slots for content-ranked unseen tail candidates.
        for name in content_items:
            candidate_ids, candidate_scores = user_content_rankings[name]
            for r in R_VALUES:
                model_name = f"{name}_r{r}"
                if not u_has_pos[user] or not profile_has_signal[name][user]:
                    primary = als_order[:K-r].astype(int).tolist()
                    injected = []
                else:
                    primary = als_order[:K-r].astype(int).tolist()
                    excluded = np.concatenate((seen, np.asarray(primary, dtype=np.int32)))
                    injected = _top_r_content_candidates(
                        candidate_ids, candidate_scores, excluded, r
                    )
                slate = _fill_from_als(primary, injected, als_order, K)
                if len(slate) != K or len(set(slate)) != K:
                    raise AssertionError(f"Bad slate: user={user}, model={model_name}, slate={slate}")
                if any(item in seen_set for item in slate):
                    raise AssertionError(f"Seen item found in slate: user={user}, model={model_name}")
                if any(not tail_mask[item] for item in injected):
                    raise AssertionError(f"Reserved item is not in tail pool: user={user}, model={model_name}")
                batch_slates[model_name].append(slate)
                batch_injected[model_name].append(injected)

        # Candidate-specific slates are aligned with the current user index.

    # Build metrics and diagnostics in candidate order, aligned with batch_users.
    for model_name in CANDIDATES:
        metric_rows = []
        diagnostic_rows = []
        for row_idx, user_value in enumerate(batch_users):
            user = int(user_value)
            slate = batch_slates[model_name][row_idx]
            injected = batch_injected[model_name][row_idx]
            slate_set = set(slate)
            als_top10_set = set(als_orders[row_idx][:K].astype(int).tolist())
            overall_rel = relevant_by_scope["overall"].get(user, set())
            tail_rel = relevant_by_scope["tail"].get(user, set())
            strict_cold_rel = relevant_by_scope["strict_cold"].get(user, set())
            warm_tail_rel = relevant_by_scope["warm_tail"].get(user, set())

            row_metrics = {}
            for scope, rel in [
                ("overall", overall_rel),
                ("tail", tail_rel),
                ("strict_cold", strict_cold_rel),
                ("warm_tail", warm_tail_rel),
            ]:
                sm = _metrics_for_recommendation(slate, rel)
                for metric, value in sm.items():
                    row_metrics[f"{scope}_{metric}"] = value
            metric_rows.append(row_metrics)

            displaced = als_top10_set - slate_set
            # For no-history users, the content policy is intentionally bypassed.
            if model_name == "ALS_only":
                requested_slots = 0
                content_slots_filled = 0
                fallback_slots = 0
            elif model_name == "ALS_tail_r1":
                requested_slots = 1 if u_has_pos[user] else 0
                content_slots_filled = 0
                fallback_slots = max(requested_slots - len(injected), 0)
            else:
                requested_slots = int(model_name.rsplit("r", 1)[1])
                content_slots_filled = len(injected)
                fallback_slots = max(requested_slots - content_slots_filled, 0) if u_has_pos[user] else 0

            diagnostic_rows.append({
                "tail_items_in_final_slate": int(sum(tail_mask[item] for item in slate)),
                "tail_share": float(sum(tail_mask[item] for item in slate) / K),
                "reserved_slots_requested": requested_slots,
                "reserved_items_injected": len(injected),
                "content_slots_filled": content_slots_filled,
                "fallback_slots_used": fallback_slots,
                "no_positive_history_user": int(not u_has_pos[user]),
                "content_profile_has_signal": int(
                    profile_has_signal[model_name.split("_r")[0]][user]
                ) if model_name.startswith(("content_v1", "tmdb_v2")) else np.nan,
                "injected_relevant_overall": int(sum(item in overall_rel for item in injected)),
                "injected_relevant_tail": int(sum(item in tail_rel for item in injected)),
                "displaced_relevant_als_overall": int(sum(item in overall_rel for item in displaced)),
                "displaced_relevant_als_tail": int(sum(item in tail_rel for item in displaced)),
                "als_top10_relevant_overall": int(sum(item in overall_rel for item in als_top10_set)),
                "final_relevant_overall": int(sum(item in overall_rel for item in slate)),
                "als_top10_relevant_tail": int(sum(item in tail_rel for item in als_top10_set)),
                "final_relevant_tail": int(sum(item in tail_rel for item in slate)),
            })

        batch_metric_results[model_name].extend(metric_rows)
        batch_diag_results[model_name].extend(diagnostic_rows)

    batch_unique_tail = {
        model_name: {
            item for slate in batch_slates[model_name] for item in slate if tail_mask[item]
        }
        for model_name in CANDIDATES
    }
    return batch_slates, batch_injected, batch_unique_tail

print("Reserved-slot strategy and metric functions defined.")

Reserved-slot strategy and metric functions defined.


## 6 · Smoke-test the slate logic

Before running the full validation grid, create tiny example slates from a few validation users. Checks verify that all lists have 10 unique unseen items, content injections are in the tail pool, and content methods retain the ALS prefix. This smoke test does not evaluate or select models.

In [5]:
# Smoke test the recommendation builder on a few real users.
# This defines the accumulators used by the generator and performs a one-seed small batch.
_smoke_users = eval_users[:3]
_smoke_seed = 0
_smoke_rng_als = np.random.default_rng(SEED + _smoke_seed)
_smoke_rng_content = {
    "content_v1": np.random.default_rng(SEED + 1000 + _smoke_seed),
    "tmdb_v2": np.random.default_rng(SEED + 2000 + _smoke_seed),
}
# The helper appends into these per-model accumulators; they are local for the smoke test.
batch_metric_results = {name: [] for name in CANDIDATES}
batch_diag_results = {name: [] for name in CANDIDATES}
_smoke_slates, _smoke_injected, _smoke_unique = _generate_batch_slates(
    _smoke_users, _smoke_seed, _smoke_rng_als, _smoke_rng_content
)
for name in CANDIDATES:
    assert len(_smoke_slates[name]) == len(_smoke_users)
    for user, slate in zip(_smoke_users, _smoke_slates[name]):
        assert len(slate) == K and len(set(slate)) == K
        assert not set(slate).intersection(_seen_items_for_user(int(user)))
    print(f"{name:16s} | smoke test passed")
print("Smoke test passed; full evaluation is not run yet.")

ALS_only         | smoke test passed
ALS_tail_r1      | smoke test passed
content_v1_r1    | smoke test passed
content_v1_r2    | smoke test passed
content_v1_r3    | smoke test passed
tmdb_v2_r1       | smoke test passed
tmdb_v2_r2       | smoke test passed
tmdb_v2_r3       | smoke test passed
Smoke test passed; full evaluation is not run yet.


## 7 · Run the predefined validation grid

This is the longer cell. For each of 10 tie-breaking seeds, it creates each user's recommendation lists once per candidate and computes metrics for overall, tail, strict-cold, and warm-tail targets from those same lists. Candidate rankings are full-catalogue View A; only the relevant target set changes by scope.

The cell saves no test metrics and does not tune any feature weights. If runtime or memory is an issue, reduce `BATCH_SIZE` (for example to 48); do not change the predefined candidate grid after inspecting results.

In [6]:
# Full evaluation. This may take several minutes because it ranks the full catalogue
# for every eligible validation user, across 10 tie-breaking seeds and 8 strategies.

metric_runs = {name: [] for name in CANDIDATES}
diagnostic_runs = {name: [] for name in CANDIDATES}
unique_tail_count_by_seed = {name: [] for name in CANDIDATES}

for tie_seed in range(N_TIE_SEEDS):
    print(f"Validation tie seed {tie_seed + 1}/{N_TIE_SEEDS}")
    seed_metric_records = {name: [] for name in CANDIDATES}
    seed_diagnostic_records = {name: [] for name in CANDIDATES}
    seed_unique_tail_items = {name: set() for name in CANDIDATES}
    rng_als = np.random.default_rng(SEED + tie_seed)
    rng_content = {
        "content_v1": np.random.default_rng(SEED + 1000 + tie_seed),
        "tmdb_v2": np.random.default_rng(SEED + 2000 + tie_seed),
    }

    for start in range(0, len(eval_users), BATCH_SIZE):
        batch_users = eval_users[start:start + BATCH_SIZE]
        batch_metric_results = {name: [] for name in CANDIDATES}
        batch_diag_results = {name: [] for name in CANDIDATES}
        batch_slates, batch_injected, batch_unique_tail = _generate_batch_slates(
            batch_users, tie_seed, rng_als, rng_content
        )
        for name in CANDIDATES:
            seed_metric_records[name].extend(batch_metric_results[name])
            seed_diagnostic_records[name].extend(batch_diag_results[name])
            seed_unique_tail_items[name].update(batch_unique_tail[name])

        if (start // BATCH_SIZE + 1) % 10 == 0 or start + BATCH_SIZE >= len(eval_users):
            print(f"  processed {min(start + BATCH_SIZE, len(eval_users)):,}/{len(eval_users):,} users")

    for name in CANDIDATES:
        metric_df = pd.DataFrame(seed_metric_records[name], index=pd.Index(eval_users, name="user_id"))
        diag_df = pd.DataFrame(seed_diagnostic_records[name], index=pd.Index(eval_users, name="user_id"))
        assert len(metric_df) == len(diag_df) == len(eval_users)
        metric_runs[name].append(metric_df)
        diagnostic_runs[name].append(diag_df)
        unique_tail_count_by_seed[name].append(len(seed_unique_tail_items[name]))

# Average every user's metrics and diagnostics over the 10 tie-breaking seeds.
user_metrics = {
    name: pd.concat(runs).groupby(level=0).mean().sort_index()
    for name, runs in metric_runs.items()
}
user_diagnostics = {
    name: pd.concat(runs).groupby(level=0).mean().sort_index()
    for name, runs in diagnostic_runs.items()
}

assert user_metrics["ALS_only"].index.equals(pd.Index(eval_users, name="user_id"))
assert np.isfinite(user_metrics["ALS_only"]["overall_ndcg"]).all()
print("\nFull validation grid finished.")

Validation tie seed 1/10
  processed 960/5,661 users
  processed 1,920/5,661 users
  processed 2,880/5,661 users
  processed 3,840/5,661 users
  processed 4,800/5,661 users
  processed 5,661/5,661 users
Validation tie seed 2/10
  processed 960/5,661 users
  processed 1,920/5,661 users
  processed 2,880/5,661 users
  processed 3,840/5,661 users
  processed 4,800/5,661 users
  processed 5,661/5,661 users
Validation tie seed 3/10
  processed 960/5,661 users
  processed 1,920/5,661 users
  processed 2,880/5,661 users
  processed 3,840/5,661 users
  processed 4,800/5,661 users
  processed 5,661/5,661 users
Validation tie seed 4/10
  processed 960/5,661 users
  processed 1,920/5,661 users
  processed 2,880/5,661 users
  processed 3,840/5,661 users
  processed 4,800/5,661 users
  processed 5,661/5,661 users
Validation tie seed 5/10
  processed 960/5,661 users
  processed 1,920/5,661 users
  processed 2,880/5,661 users
  processed 3,840/5,661 users
  processed 4,800/5,661 users
  processed 5,6

## 8 · Paired confidence intervals, guardrail, and diagnostics

Bootstrap users in pairs: every candidate and ALS baseline uses the same resampled users for each evaluation scope. Because some users have no targets in the tail/cold/warm-tail scope, each scope is evaluated only on users with at least one relevant target in that scope.

The 95% intervals are validation uncertainty estimates. Multiple candidate configurations are being screened, so they should not be presented as confirmatory test-set inference.

In [7]:
def paired_bootstrap(delta_values, draw_indices):
    """Mean user-level difference and percentile bootstrap 95% CI."""
    values = np.asarray(delta_values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    # draw_indices must index this exact evaluated-user vector.
    boot_means = values[draw_indices].mean(axis=1)
    return (
        float(values.mean()),
        float(np.percentile(boot_means, 2.5)),
        float(np.percentile(boot_means, 97.5)),
    )

scope_metric_config = {
    "overall_view_a": ("overall", scope_users["overall"]),
    "tail_view_a": ("tail", scope_users["tail"]),
    "strict_cold_view_a": ("strict_cold", scope_users["strict_cold"]),
    "warm_tail_view_a": ("warm_tail", scope_users["warm_tail"]),
}
metric_comparison_rows = []
bootstrap_indices = {}
boot_rng = np.random.default_rng(SEED + 8800)
for scope_name, (scope_key, users_scope) in scope_metric_config.items():
    if len(users_scope) == 0:
        continue
    bootstrap_indices[scope_name] = boot_rng.integers(
        0, len(users_scope), size=(N_BOOT, len(users_scope)), dtype=np.int32
    )
    draw_idx = bootstrap_indices[scope_name]
    base_df = user_metrics["ALS_only"].loc[users_scope]

    for model_name in CANDIDATES:
        if model_name == "ALS_only":
            continue
        candidate_df = user_metrics[model_name].loc[users_scope]
        for metric in ("ndcg", "recall", "hit_rate"):
            col = f"{scope_key}_{metric}"
            base_values = base_df[col].to_numpy(dtype=np.float64)
            candidate_values = candidate_df[col].to_numpy(dtype=np.float64)
            if not (np.isfinite(base_values).all() and np.isfinite(candidate_values).all()):
                raise ValueError(f"Non-finite paired metric for {scope_name}/{model_name}/{metric}")
            delta = candidate_values - base_values
            point, low, high = paired_bootstrap(delta, draw_idx)
            metric_comparison_rows.append({
                "scope": scope_name,
                "model": model_name,
                "metric": metric,
                "users": len(users_scope),
                "als_mean": float(base_values.mean()),
                "candidate_mean": float(candidate_values.mean()),
                "paired_delta_vs_als": point,
                "delta_ci95_low": low,
                "delta_ci95_high": high,
            })

paired_ci_results = pd.DataFrame(metric_comparison_rows)

# Decision table: operational point-estimate guardrail + uncertainty margin.
overall_users = scope_users["overall"]
tail_users = scope_users["tail"]
base_overall = user_metrics["ALS_only"].loc[overall_users, "overall_ndcg"].to_numpy(dtype=np.float64)
base_tail = user_metrics["ALS_only"].loc[tail_users, "tail_ndcg"].to_numpy(dtype=np.float64)
als_overall_ndcg = float(base_overall.mean())
retention_floor = RETENTION_FRACTION * als_overall_ndcg

# Same bootstrap draws are reused for paired confidence intervals on the guardrail margin.
guardrail_draws = bootstrap_indices["overall_view_a"]
decision_rows = []
for model_name in CANDIDATES:
    if model_name == "ALS_only":
        continue
    overall_df = user_metrics[model_name].loc[overall_users]
    tail_df = user_metrics[model_name].loc[tail_users]
    candidate_overall_ndcg = float(overall_df["overall_ndcg"].mean())
    candidate_tail_ndcg = float(tail_df["tail_ndcg"].mean())
    overall_margin = overall_df["overall_ndcg"].to_numpy(dtype=np.float64) - RETENTION_FRACTION * base_overall
    g_point, g_low, g_high = paired_bootstrap(overall_margin, guardrail_draws)
    tail_delta = tail_df["tail_ndcg"].to_numpy(dtype=np.float64) - base_tail
    tail_delta_point, tail_delta_low, tail_delta_high = paired_bootstrap(
        tail_delta, bootstrap_indices["tail_view_a"]
    )
    diag = user_diagnostics[model_name]
    decision_rows.append({
        "model": model_name,
        "overall_ndcg_at_10": candidate_overall_ndcg,
        "overall_recall_at_10": float(overall_df["overall_recall"].mean()),
        "als_overall_ndcg_at_10": als_overall_ndcg,
        "overall_retention_floor": retention_floor,
        "overall_guardrail_pass_point_estimate": bool(candidate_overall_ndcg >= retention_floor),
        "guardrail_margin_G": g_point,
        "guardrail_G_ci95_low": g_low,
        "guardrail_G_ci95_high": g_high,
        "tail_ndcg_at_10": candidate_tail_ndcg,
        "als_tail_ndcg_at_10": float(base_tail.mean()),
        "tail_delta_vs_als": tail_delta_point,
        "tail_delta_ci95_low": tail_delta_low,
        "tail_delta_ci95_high": tail_delta_high,
        "strict_cold_ndcg_at_10": float(user_metrics[model_name].loc[scope_users["strict_cold"], "strict_cold_ndcg"].mean()) if len(scope_users["strict_cold"]) else np.nan,
        "warm_tail_ndcg_at_10": float(user_metrics[model_name].loc[scope_users["warm_tail"], "warm_tail_ndcg"].mean()) if len(scope_users["warm_tail"]) else np.nan,
        "mean_reserved_items_injected": float(diag["reserved_items_injected"].mean()),
        "mean_fallback_slots_used": float(diag["fallback_slots_used"].mean()),
    })

decision_table = pd.DataFrame(decision_rows).sort_values(
    ["overall_guardrail_pass_point_estimate", "tail_ndcg_at_10"], ascending=[False, False]
).reset_index(drop=True)

# Diagnostics summary, including relevance gained/lost relative to ALS.
diagnostic_rows = []
for model_name in CANDIDATES:
    diag = user_diagnostics[model_name]
    avg_metrics = user_metrics[model_name]
    base_metrics = user_metrics["ALS_only"]
    overall_hit_diff = (
        avg_metrics.loc[overall_users, "overall_hit_count"].to_numpy(dtype=np.float64)
        - base_metrics.loc[overall_users, "overall_hit_count"].to_numpy(dtype=np.float64)
    )
    tail_hit_diff = (
        avg_metrics.loc[tail_users, "tail_hit_count"].to_numpy(dtype=np.float64)
        - base_metrics.loc[tail_users, "tail_hit_count"].to_numpy(dtype=np.float64)
    )
    diagnostic_rows.append({
        "model": model_name,
        "mean_tail_items_per_list": float(diag["tail_items_in_final_slate"].mean()),
        "mean_tail_share_of_slots": float(diag["tail_share"].mean()),
        "unique_tail_movies_recommended_mean_across_seeds": float(np.mean(unique_tail_count_by_seed[model_name])),
        "unique_tail_movies_recommended_min": int(np.min(unique_tail_count_by_seed[model_name])),
        "unique_tail_movies_recommended_max": int(np.max(unique_tail_count_by_seed[model_name])),
        "mean_reserved_items_injected": float(diag["reserved_items_injected"].mean()),
        "mean_injected_relevant_overall_per_user": float(diag["injected_relevant_overall"].mean()),
        "mean_injected_relevant_tail_per_user": float(diag["injected_relevant_tail"].mean()),
        "mean_displaced_relevant_als_overall_per_user": float(diag["displaced_relevant_als_overall"].mean()),
        "mean_displaced_relevant_als_tail_per_user": float(diag["displaced_relevant_als_tail"].mean()),
        "users_gaining_overall_hits_vs_als": int((overall_hit_diff > 1e-9).sum()),
        "users_losing_overall_hits_vs_als": int((overall_hit_diff < -1e-9).sum()),
        "users_same_overall_hits_vs_als": int((np.abs(overall_hit_diff) <= 1e-9).sum()),
        "users_gaining_tail_hits_vs_als": int((tail_hit_diff > 1e-9).sum()),
        "users_losing_tail_hits_vs_als": int((tail_hit_diff < -1e-9).sum()),
        "users_same_tail_hits_vs_als": int((np.abs(tail_hit_diff) <= 1e-9).sum()),
        "users_without_positive_history_fraction": float(diag["no_positive_history_user"].mean()),
    })

diagnostic_summary = pd.DataFrame(diagnostic_rows)

print("PAIRED VALIDATION COMPARISONS VS ALS")
display(paired_ci_results.round(6))
print("\nRESERVED-SLOT DECISION TABLE")
display(decision_table.round(6))
print("\nOPERATIONAL DIAGNOSTICS")
display(diagnostic_summary.round(6))

# Save compact reports and an optional per-user table for follow-up diagnostics.
REPORT_DIR = _PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
paired_path = REPORT_DIR / "reserved_slot_validation_paired_ci.csv"
decision_path = REPORT_DIR / "reserved_slot_validation_decision.csv"
diagnostic_path = REPORT_DIR / "reserved_slot_validation_diagnostics.csv"
user_metrics_path = REPORT_DIR / "reserved_slot_validation_user_metrics.csv"
paired_ci_results.to_csv(paired_path, index=False)
decision_table.to_csv(decision_path, index=False)
diagnostic_summary.to_csv(diagnostic_path, index=False)

user_metric_export = pd.concat(
    {name: frame for name, frame in user_metrics.items()},
    names=["model", "user_id"],
).reset_index()
user_metric_export.to_csv(user_metrics_path, index=False)

print("\nSaved reports:")
print(paired_path)
print(decision_path)
print(diagnostic_path)
print(user_metrics_path)
print("\nTest set used: No")
print(f"ALS overall View A NDCG@10: {als_overall_ndcg:.6f}")
print(f"98% overall retention floor: {retention_floor:.6f}")

eligible = decision_table.loc[decision_table["overall_guardrail_pass_point_estimate"]]
if len(eligible):
    selected = eligible.sort_values("tail_ndcg_at_10", ascending=False).iloc[0]
    print("\nBEST RESERVED-SLOT CANDIDATE THAT PASSES THE POINT-ESTIMATE GUARDRAIL")
    print(f"Model: {selected['model']}")
    print(f"Tail View A NDCG@10: {selected['tail_ndcg_at_10']:.6f}")
    print(f"Overall View A NDCG@10: {selected['overall_ndcg_at_10']:.6f}")
    print(
        "Guardrail margin G 95% CI: "
        f"[{selected['guardrail_G_ci95_low']:+.6f}, {selected['guardrail_G_ci95_high']:+.6f}]"
    )
    print("Remember: the guardrail pass is a point-estimate selection rule; a CI crossing zero means preservation is uncertain.")
else:
    print("\nNO RESERVED-SLOT CANDIDATE PASSES THE 98% OVERALL NDCG POINT-ESTIMATE GUARDRAIL.")
    print("Retain ALS-only and do not promote a reserved-slot model from this grid.")

PAIRED VALIDATION COMPARISONS VS ALS


,scope,model,metric,users,als_mean,candidate_mean,paired_delta_vs_als,delta_ci95_low,delta_ci95_high
0,overall_view_a,ALS_tail_r1,ndcg,5661,0.081462,0.078520,-0.002942,-0.003479,-0.002435
1,overall_view_a,ALS_tail_r1,recall,5661,0.106220,0.099434,-0.006785,-0.008210,-0.005442
2,overall_view_a,ALS_tail_r1,hit_rate,5661,0.302067,0.289525,-0.012542,-0.015898,-0.009358
3,overall_view_a,content_v1_r1,ndcg,5661,0.081462,0.078256,-0.003206,-0.003715,-0.002717
4,overall_view_a,content_v1_r1,recall,5661,0.106220,0.098843,-0.007377,-0.008792,-0.006069
...,...,...,...,...,...,...,...,...,...
79,warm_tail_view_a,tmdb_v2_r2,recall,931,0.000000,0.002184,0.002184,0.000000,0.004874
80,warm_tail_view_a,tmdb_v2_r2,hit_rate,931,0.000000,0.003867,0.003867,0.000000,0.008383
81,warm_tail_view_a,tmdb_v2_r3,ndcg,931,0.000000,0.001311,0.001311,0.000310,0.002552
82,warm_tail_view_a,tmdb_v2_r3,recall,931,0.000000,0.003759,0.003759,0.000895,0.007698



RESERVED-SLOT DECISION TABLE


,model,overall_ndcg_at_10,overall_recall_at_10,als_overall_ndcg_at_10,overall_retention_floor,overall_guardrail_pass_point_estimate,guardrail_margin_G,guardrail_G_ci95_low,guardrail_G_ci95_high,tail_ndcg_at_10,als_tail_ndcg_at_10,tail_delta_vs_als,tail_delta_ci95_low,tail_delta_ci95_high,strict_cold_ndcg_at_10,warm_tail_ndcg_at_10,mean_reserved_items_injected,mean_fallback_slots_used
0,ALS_tail_r1,0.078520,0.099434,0.081462,0.079833,False,-0.001313,-0.001836,-0.000782,0.003720,0.0,0.003720,0.002036,0.005850,0.0,0.003788,1.0,0.0
1,tmdb_v2_r3,0.070710,0.082494,0.081462,0.079833,False,-0.009123,-0.010065,-0.008143,0.001143,0.0,0.001143,0.000290,0.002199,0.0,0.001311,3.0,0.0
2,content_v1_r3,0.070669,0.082381,0.081462,0.079833,False,-0.009164,-0.010071,-0.008190,0.000984,0.0,0.000984,0.000268,0.001972,0.0,0.001053,3.0,0.0
3,content_v1_r2,0.074633,0.090859,0.081462,0.079833,False,-0.005200,-0.005916,-0.004444,0.000743,0.0,0.000743,0.000183,0.001595,0.0,0.000807,2.0,0.0
4,tmdb_v2_r2,0.074644,0.090907,0.081462,0.079833,False,-0.005188,-0.005909,-0.004444,0.000722,0.0,0.000722,0.000086,0.001590,0.0,0.000761,2.0,0.0
5,tmdb_v2_r1,0.078300,0.098953,0.081462,0.079833,False,-0.001533,-0.002044,-0.001042,0.000611,0.0,0.000611,0.000000,0.001408,0.0,0.000647,1.0,0.0
6,content_v1_r1,0.078256,0.098843,0.081462,0.079833,False,-0.001576,-0.002072,-0.001088,0.000334,0.0,0.000334,0.000000,0.000963,0.0,0.000340,1.0,0.0



OPERATIONAL DIAGNOSTICS


,model,mean_tail_items_per_list,mean_tail_share_of_slots,unique_tail_movies_recommended_mean_across_seeds,unique_tail_movies_recommended_min,unique_tail_movies_recommended_max,mean_reserved_items_injected,mean_injected_relevant_overall_per_user,mean_injected_relevant_tail_per_user,mean_displaced_relevant_als_overall_per_user,mean_displaced_relevant_als_tail_per_user,users_gaining_overall_hits_vs_als,users_losing_overall_hits_vs_als,users_same_overall_hits_vs_als,users_gaining_tail_hits_vs_als,users_losing_tail_hits_vs_als,users_same_tail_hits_vs_als,users_without_positive_history_fraction
0,ALS_only,0.0,0.0,0.0,0,0,0.0,0.000000,0.000000,0.000000,0.0,0,0,5661,0,0,948,0.0
1,ALS_tail_r1,1.0,0.1,715.0,715,715,1.0,0.002826,0.002826,0.029677,0.0,16,168,5477,16,0,932,0.0
2,content_v1_r1,1.0,0.1,1110.7,1098,1122,1.0,0.000212,0.000212,0.029677,0.0,3,168,5490,3,0,945,0.0
3,content_v1_r2,2.0,0.2,1740.1,1725,1754,2.0,0.000742,0.000742,0.061827,0.0,9,341,5311,9,0,939,0.0
4,content_v1_r3,3.0,0.3,2201.0,2188,2223,3.0,0.000901,0.000901,0.094506,0.0,9,507,5145,9,0,939,0.0
5,tmdb_v2_r1,1.0,0.1,175.6,174,177,1.0,0.000530,0.000530,0.029677,0.0,3,168,5490,3,0,945,0.0
6,tmdb_v2_r2,2.0,0.2,296.6,296,297,2.0,0.000636,0.000636,0.061827,0.0,3,340,5318,4,0,944,0.0
7,tmdb_v2_r3,3.0,0.3,432.0,432,432,3.0,0.001060,0.001060,0.094506,0.0,5,506,5150,6,0,942,0.0



Saved reports:
C:\Users\Lenovo\Documents\movie-recommender\reports\reserved_slot_validation_paired_ci.csv
C:\Users\Lenovo\Documents\movie-recommender\reports\reserved_slot_validation_decision.csv
C:\Users\Lenovo\Documents\movie-recommender\reports\reserved_slot_validation_diagnostics.csv
C:\Users\Lenovo\Documents\movie-recommender\reports\reserved_slot_validation_user_metrics.csv

Test set used: No
ALS overall View A NDCG@10: 0.081462
98% overall retention floor: 0.079833

NO RESERVED-SLOT CANDIDATE PASSES THE 98% OVERALL NDCG POINT-ESTIMATE GUARDRAIL.
Retain ALS-only and do not promote a reserved-slot model from this grid.


## 9 · How to read the results

Use `reserved_slot_validation_decision.csv` as the compact decision summary and `reserved_slot_validation_paired_ci.csv` for uncertainty. The best reserved-slot candidate is selected **only among candidates meeting the overall 98% point-estimate guardrail**, maximizing tail View A NDCG@10. The confidence interval for `G = candidate overall NDCG@10 − 0.98 × ALS overall NDCG@10` is reported separately; if it crosses zero, preservation is uncertain.

Check the diagnostics before celebrating a tail lift:

- Did injected candidates actually hit held-out relevant items?
- How many relevant ALS recommendations were displaced?
- Did more users gain hits than lose them?
- Did tail exposure and unique tail-item coverage increase, or did recommendations concentrate on a few movies?
- How often did the content profile fail to provide the requested number of slots and trigger ALS fallback?
- Do strict-cold results differ from warm-tail results?

**Exposure is not relevance.** A reserved slot guarantees an eligible rare movie appears, not that it is the right movie. If no candidate passes the guardrail, the evidence says not to deploy a fixed reserved-slot hybrid under this configuration. Do not tune on the test set; decide whether to investigate a learned hybrid (e.g., LightFM with item features) only after reviewing this controlled diagnostic.